<a href="https://colab.research.google.com/github/Pinuig/machine-learning-lab/blob/main/TrashDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub Pillow -q

import os
import kagglehub
from PIL import Image

print("Downloading Trash Dataset...")
# Download Garbage Classification Dataset via Kagglehub
download_path = kagglehub.dataset_download("mostafaabla/garbage-classification")

TARGET_DIR = '/content/trash_dataset/cropped_dataset'
TARGET_CLASSES = ['plastic', 'metal', 'paper']
TARGET_SIZE = (160, 120)  # (Width = 160, Height = 120)

os.makedirs(TARGET_DIR, exist_ok=True)

processed_counts = {c: 0 for c in TARGET_CLASSES}

print("Resizing all images to 160x120 pixels...")

for root, dirs, files in os.walk(download_path):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
            parent_dir = os.path.basename(root).lower()

            matched_class = None
            for target in TARGET_CLASSES:
                if target in parent_dir:
                    matched_class = target
                    break

            if matched_class:
                class_dest_dir = os.path.join(TARGET_DIR, matched_class)
                os.makedirs(class_dest_dir, exist_ok=True)

                src_file_path = os.path.join(root, file)
                dest_file_path = os.path.join(class_dest_dir, file)

                try:
                    with Image.open(src_file_path) as img:
                        img = img.convert('RGB')
                        # Resize directly to 160x120 pixels using high-quality Lanczos resampling
                        img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
                        img_resized.save(dest_file_path)
                        processed_counts[matched_class] += 1
                except Exception as e:
                    pass

print("\n--------------------------------------------------")
print("✅ DATASET PROCESSING COMPLETE!")
print(f"Target Directory: '{TARGET_DIR}'")
print(f"Image Size: {TARGET_SIZE[0]} x {TARGET_SIZE[1]} pixels")
print("Images Processed Per Category:")
for cat, count in processed_counts.items():
    print(f"  - {cat}: {count} images")
print("--------------------------------------------------")

Using Colab cache for faster access to the 'garbage-classification' dataset.
Resizing all images to 160x120 pixels...

--------------------------------------------------
✅ DATASET PROCESSING COMPLETE!
Target Directory: '/content/trash_dataset/cropped_dataset'
Image Size: 160 x 120 pixels
Images Processed Per Category:
  - plastic: 865 images
  - metal: 769 images
  - paper: 1050 images
--------------------------------------------------


In [ ]:

import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

DATA_DIR = '/content/trash_dataset/cropped_dataset'
IMG_HEIGHT = 120
IMG_WIDTH = 160
BATCH_SIZE = 32

print("Loading dataset from:", DATA_DIR)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"\n✅ SUCCESS: Detected {NUM_CLASSES} Classes: {class_names}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet',
    alpha=0.35
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- PHASE 1: Feature Extraction ---")
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stopping]
)

print("\n--- PHASE 2: Fine-Tuning ---")
base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stopping]
)

model.save('trash_classifier_model.keras')

print("\n--- INT8 Quantization ---")
def representative_data_gen():
    for input_batch, _ in train_ds.take(20):
        for img in input_batch:
            yield [tf.expand_dims(tf.cast(img, tf.float32), axis=0)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8


converter._experimental_disable_per_channel_quantization_for_dense_layers = True

tflite_model_quant = converter.convert()

tflite_file = 'trash_classifier_quant.tflite'
with open(tflite_file, 'wb') as f:
    f.write(tflite_model_quant)

print(f"\nQuantized TFLite model saved as '{tflite_file}' ({len(tflite_model_quant) / 1024:.2f} KB)")


header_content = f"""#ifndef MODEL_DATA_H_
#define MODEL_DATA_H_

#include <cstdint>

// Model byte size: {len(tflite_model_quant)} bytes
alignas(8) const unsigned char trash_classifier_quant_tflite[] = {{
"""

hex_bytes = [f"0x{b:02x}" for b in tflite_model_quant]
for i in range(0, len(hex_bytes), 12):
    header_content += "  " + ", ".join(hex_bytes[i:i+12]) + ",\n"

header_content += "};\n\n"
header_content += f"const unsigned int trash_classifier_quant_tflite_len = {len(tflite_model_quant)};\n\n"
header_content += "#endif  // MODEL_DATA_H_\n"

with open('model_data.h', 'w') as f:
    f.write(header_content)

with open('labels.h', 'w') as f:
    f.write('#ifndef LABELS_H\n#define LABELS_H\n\n')
    f.write(f'const int NUM_LABELS = {NUM_CLASSES};\n')
    f.write('const char* LABELS[] = {\n')
    for name in class_names:
        f.write(f'    "{name}",\n')
    f.write('};\n\n#endif // LABELS_H\n')

# Automatically download generated headers to your PC
from google.colab import files
files.download('model_data.h')
files.download('labels.h')

print("\n--------------------------------------------------")
print("🎉 TRAINING & EXPORT COMPLETE!")
print("  - model_data.h and labels.h downloaded automatically.")
print("--------------------------------------------------")

Loading dataset from: /content/trash_dataset/cropped_dataset
Found 2684 files belonging to 3 classes.
Using 2148 files for training.
Found 2684 files belonging to 3 classes.
Using 536 files for validation.

✅ SUCCESS: Detected 3 Classes: ['metal', 'paper', 'plastic']


/tmp/ipykernel_3177/4288017926.py:46: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(



--- PHASE 1: Feature Extraction ---
Epoch 1/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.6774 - loss: 0.7959 - val_accuracy: 0.8228 - val_loss: 0.4410
Epoch 2/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7938 - loss: 0.5297 - val_accuracy: 0.8116 - val_loss: 0.4890
Epoch 3/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.8101 - loss: 0.4895 - val_accuracy: 0.8545 - val_loss: 0.3750
Epoch 4/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.8236 - loss: 0.4417 - val_accuracy: 0.8787 - val_loss: 0.3337
Epoch 5/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.8310 - loss: 0.4198 - val_accuracy: 0.8731 - val_loss: 0.3211
Epoch 6/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.8459 - loss: 0.3951 - val_accuracy: 0.8675 - val_loss: 0.3192
Epoch 7/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8627 - loss: 0.3479 - val_accuracy: 0.8955 - val_loss: 0.2783
Epoch 8/20
68/68 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.8678 - l

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



Quantized TFLite model saved as 'trash_classifier_quant.tflite' (611.22 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--------------------------------------------------
🎉 TRAINING & EXPORT COMPLETE!
  - model_data.h and labels.h downloaded automatically.
--------------------------------------------------


In [ ]:
import tensorflow as tf
import numpy as np

# Get input shape from your trained model in memory
input_shape = model.input_shape
img_height, img_width = input_shape[1], input_shape[2]
img_channels = input_shape[3] if len(input_shape) == 4 else 3

# Representative dataset generator for INT8 calibration
def representative_data_gen():
    for _ in range(100):
        data = np.random.uniform(-1.0, 1.0, size=(1, img_height, img_width, img_channels)).astype(np.float32)
        yield [data]

# Configure converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# CRITICAL ESP32-CAM FIX
converter._experimental_disable_per_channel_quantization_for_dense_layers = True

# Convert and save model_data.h
tflite_quant_model = converter.convert()

header_content = f"""#ifndef MODEL_DATA_H_
#define MODEL_DATA_H_

#include <cstdint>

alignas(8) const unsigned char trash_classifier_quant_tflite[] = {{
"""

hex_bytes = [f"0x{b:02x}" for b in tflite_quant_model]
for i in range(0, len(hex_bytes), 12):
    header_content += "  " + ", ".join(hex_bytes[i:i+12]) + ",\n"

header_content += f"}};\n\nconst unsigned int trash_classifier_quant_tflite_len = {len(tflite_quant_model)};\n\n#endif  // MODEL_DATA_H_\n"

with open("model_data.h", "w") as f:
    f.write(header_content)

from google.colab import files
files.download("model_data.h")

Saved artifact at '/tmp/tmp1xwbh5hf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 120, 160, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  134517943542480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134515279521040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134515279520848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134515279519696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134515279521616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134515279519120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134515279522192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134515279523152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134515279522768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134515279521232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345152795

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>